In [23]:
import re 
import pandas as pd 
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sentence_transformers import SentenceTransformer
import os 

In [2]:
def normalize(text):
    text = re.sub(r"[^가-힣0-9a-zA-Z\s\.]", ' ', text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [3]:
file_path = "data/가전/"

file_list = os.listdir(file_path)

df = pd.DataFrame()

for file in file_list:
    data = pd.read_json(file_path + file)
    df = pd.concat( [df, data], axis= 0 )
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Index            4056 non-null   int64  
 1   RawText          4056 non-null   object 
 2   Source           4056 non-null   object 
 3   Domain           4056 non-null   object 
 4   MainCategory     4056 non-null   object 
 5   ProductName      4056 non-null   object 
 6   ReviewScore      4056 non-null   int64  
 7   Syllable         4056 non-null   int64  
 8   Word             4056 non-null   int64  
 9   RDate            4056 non-null   int64  
 10  GeneralPolarity  3678 non-null   float64
 11  Aspects          4056 non-null   object 
dtypes: float64(1), int64(5), object(6)
memory usage: 411.9+ KB


In [4]:
df = df[['RawText', 'GeneralPolarity']]

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4056 entries, 0 to 99
Data columns (total 2 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   RawText          4056 non-null   object 
 1   GeneralPolarity  3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 95.1+ KB


In [6]:
df['RawText'] = df['RawText'].map(normalize)

In [8]:
df = df.loc[df['RawText'].str.len() > 1, ]

In [10]:
df.drop_duplicates('RawText', inplace=True)

In [12]:
df.rename(columns = {
    'GeneralPolarity' : 'label'
}, inplace=True)

In [14]:
na_df = df.loc[df['label'].isna()]

In [15]:
df = df.loc[~df['label'].isna()]

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 3678 entries, 0 to 99
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   RawText  3678 non-null   object 
 1   label    3678 non-null   float64
dtypes: float64(1), object(1)
memory usage: 86.2+ KB


In [20]:
df['label'] = df['label'].map({
    -1 : 0, 
    0 : 0, 
    1 : 1
})

In [21]:
df['label'].value_counts()

label
1    2220
0    1458
Name: count, dtype: int64

In [22]:
train_df, test_df = train_test_split(
    df, test_size = 0.2, random_state = 42, stratify=df['label']
)

In [36]:
train_df['label'].value_counts()

label
1    1776
0    1166
Name: count, dtype: int64

In [24]:

model_name = "BM-K/KoSimCSE-roberta-multitask"


sbert = SentenceTransformer(model_name)


No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [43]:
# 3) Dataset 정의 ---------------------------------------------------
class SBERTDataset(Dataset):
    def __init__(self, texts, labels ):
        self.texts = texts
        self.labels = labels
        print(len(self.texts), len(self.labels))

    def __len__(self): 
        return len(self.labels)
    def __getitem__(self, idx):
        with torch.inference_mode():
            embs = sbert.encode(
                self.texts[idx], convert_to_tensor=True, normalize_embeddings=normalize
            )
        labels = torch.tensor(self.labels[idx], dtype=torch.long)
        return embs, labels

In [44]:
train_ds = SBERTDataset(train_df["RawText"].tolist(), train_df["label"].tolist())
test_ds  = SBERTDataset(test_df["RawText"].tolist(),  test_df["label"].tolist())

train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)
test_dl  = DataLoader(test_ds, batch_size=256)

2942 2942
736 736


In [45]:
train_ds

In [46]:

# 4) MLP 분류기 -----------------------------------------------------
class MLPHead(nn.Module):
    def __init__(self, in_dim, hidden=256, num_classes=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    def forward(self, x): return self.net(x)

in_dim = sbert.get_sentence_embedding_dimension()
clf = MLPHead(in_dim)

crit = nn.CrossEntropyLoss()
opt = torch.optim.AdamW(clf.parameters(), lr=2e-4)


In [47]:

# 5) 학습 -----------------------------------------------------------
clf.train()
for epoch in range(5):
    total = 0.0
    for xb, yb in train_dl:
        xb, yb = xb, yb
        opt.zero_grad()
        logits = clf(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        total += loss.item() * xb.size(0)
    print(f"epoch {epoch+1}, loss={total/len(train_ds):.4f}")


RuntimeError: Tensor for argument weight is on cpu but expected on mps